# Tokens and Byte-Pair Encoding

Back in Part 1 we said an LLM predicts, one *token* at a time, what comes next. But what exactly is a token? In this part we open up that black box: why LLMs don't work directly on words or on individual characters, and how they build their vocabulary of tokens using an algorithm called **Byte-Pair Encoding (BPE)**. We'll write a tiny BPE trainer ourselves on a toy example, then look at how a real, production tokenizer splits actual text.

Looking back at what we've done so far, the *prompt* our program actually sends to the LLM is bigger than it might seem. It's not just the system and user instructions: in Part 2 it also contained the whole text of the book we were analyzing, plus the description of every tool the agent was allowed to call; in Part 1's "Interactions" section it also specified the exact format we wanted the answer in (pure JSON); and since the LLM itself remembers nothing from one call to the next (the "amnesia" we saw in Part 2), any conversation history also has to be sent again, every time, as part of the prompt. All of this — instructions, context, tool descriptions, formatting requirements, history — has to be turned into tokens before the model can read any of it. So we're going to start by studying tokenization itself.

## Why not words, why not characters?

<img src="images/typesetting.jpg" width="150" alt="Typesetting letters" style="float: left; margin-right: 15px; margin-bottom: 10px;">

An LLM needs to turn text into a sequence of discrete symbols it can work with. Two obvious choices both have problems:

* **Whole words as symbols**: the vocabulary would need an entry for every word in every language, including rare words, typos, product names, etc. That's an enormous, ever-growing vocabulary, and the model still fails completely on any word it has never seen before.
* **Individual characters as symbols**: the vocabulary stays tiny (a few hundred symbols), and any string can be represented. But sequences become very long — a short sentence becomes dozens of symbols — which makes it harder and more expensive for the model to relate symbols that are far apart.

**Tokens** (subword units) are the middle ground: common words end up as a single token, while rare or long words get split into a handful of frequent pieces. This keeps the vocabulary at a reasonable size (tens of thousands of entries) while still being able to represent *any* string, since individual characters are always available as a fallback.

## Byte-Pair Encoding (BPE): the algorithm

<img src="images/bpe_merge.jpg" width="150" alt="BPE merging process" style="float: left; margin-right: 15px; margin-bottom: 10px;">

BPE builds this middle-ground vocabulary automatically from a training corpus, by repeatedly merging the most frequent pair of adjacent symbols:

1. Start with a vocabulary made of every individual character (or byte) appearing in the corpus, and represent every word as a sequence of these characters.
2. Count how often every pair of adjacent symbols occurs across the whole corpus.
3. Take the single most frequent pair, merge it into one new symbol, and add that symbol to the vocabulary.
4. Repeat steps 2-3 a fixed number of times (this number of merges is chosen when training the tokenizer — real tokenizers typically do tens of thousands of merges).

The final vocabulary is the original characters plus every symbol created by a merge along the way. To tokenize new text, you just apply the same merges, in the same order, to it.

Let's see this in action on the small toy corpus from the original BPE paper (Sennrich et al., 2016): the words `low`, `lower`, `newest` and `widest`, each with a frequency, and a special `</w>` symbol marking the end of a word (so the tokenizer can tell "est" at the end of a word from "est" at the start of one).

In [ ]:
# Program 1: a tiny BPE trainer on a toy corpus

from collections import Counter  # Counter counts occurrences of items (like a frequency dictionary)

# The corpus represents our training data.
# Each word is split into individual characters, plus a special '</w>' marker for the end of the word.
# The integer value is the frequency of that word in the corpus.
corpus = {
    ("l", "o", "w", "</w>"): 5,
    ("l", "o", "w", "e", "r", "</w>"): 2,
    ("n", "e", "w", "e", "s", "t", "</w>"): 6,
    ("w", "i", "d", "e", "s", "t", "</w>"): 3,
}

def get_pair_counts(corpus):
    """Count how often each pair of adjacent symbols occurs, across the whole corpus."""
    counts = Counter()
    
    # We iterate over the corpus dictionary.
    # .items() returns pairs of (key, value) -> (word_tuple, frequency)
    for word, freq in corpus.items():
        # 'zip(word, word[1:])' is a Python trick to get all adjacent pairs of symbols.
        # e.g., if word is ('l', 'o', 'w', '</w>'), then word[1:] is ('o', 'w', '</w>').
        # zip() pairs them element-by-element:
        # ('l', 'o'), ('o', 'w'), ('w', '</w>').
        for a, b in zip(word, word[1:]):
            # We add the word's frequency to the pair's count
            counts[(a, b)] += freq
            
    return counts

def merge_pair(pair, corpus):
    """Replace every occurrence of `pair` by its merged symbol in every word."""
    a, b = pair
    merged = a + b  # Concatenate the two symbols, e.g., 'e' + 's' -> 'es'
    new_corpus = {}
    
    for word, freq in corpus.items():
        new_word = []
        i = 0
        while i < len(word):
            # If we find the pair 'a' and 'b' adjacent, we merge them into one
            if i < len(word) - 1 and word[i] == a and word[i + 1] == b:
                new_word.append(merged)
                i += 2  # Skip both characters since they are merged
            else:
                new_word.append(word[i])
                i += 1  # Keep the character and move to the next one
        # Save the updated word tuple in the new corpus
        new_corpus[tuple(new_word)] = freq
        
    return new_corpus

# We initialize the vocabulary with all unique individual characters present in the corpus.
# This uses a nested generator inside set(): it loops over every word, then over every symbol in that word.
vocab = set(symbol for word in corpus for symbol in word)
num_merges = 8  # Number of times we will merge the most frequent pair

for i in range(num_merges):
    pairs = get_pair_counts(corpus)
    if not pairs:
        break
        
    # 'max(pairs, key=pairs.get)' finds the key (pair) that has the maximum value (count) in the dictionary.
    # pairs.get is a function that returns the count for a given pair.
    best_pair = max(pairs, key=pairs.get)
    
    # Merge the best pair in the corpus
    corpus = merge_pair(best_pair, corpus)
    
    # Add the merged symbol to our vocabulary (e.g., 'es')
    # "".join(best_pair) joins the tuple ('e', 's') into a single string "es"
    vocab.add("".join(best_pair))
    
    # '!r' in the f-string forces Python to call repr(), which prints quotes around the merged string.
    print(f"Merge {i + 1}: {best_pair} -> {best_pair[0] + best_pair[1]!r}  (seen {pairs[best_pair]} times)")

print("\nWords after all merges:")
for word, freq in corpus.items():
    print(f"  {word}  (frequency {freq})")

print(f"\nVocabulary size: {len(vocab)} symbols")

Watch how the merges build up meaningful pieces step by step: `e`+`s` merges into `es` (it appears in both "newest" and "widest"), then `es`+`t` into `est`, then `est`+`</w>` into a whole "est" ending. Separately, `l`+`o`+`w` merges into `low`. And because "newest" is frequent enough on its own, it eventually merges all the way into a single `newest</w>` token — while "widest", which is rarer, stays split into `w`, `i`, `d`, `est</w>`.

This is exactly the trade-off we described above: frequent words tend to become a single token, while rarer ones stay broken up into smaller, more frequent pieces.

<br>
<img src="images/bpe_academic.jpg" width="500" alt="BPE Merge Steps Diagram" style="display: block; margin: 15px auto;">
<br>

## Real tokenizers: OpenAI, Mistral, Hugging Face

<img src="images/tokenizers.jpg" width="150" alt="Standard tokenizers" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Real tokenizers are trained the exact same way, just on a much bigger corpus and with tens of thousands of merges instead of 8. When you call an LLM API, you don't train this yourself: the provider ships a fixed, pre-trained vocabulary (and merge table), and you just reuse it to split your text into tokens.

Let's compare three of them on the very same sentence:
* [`tiktoken`](https://github.com/openai/tiktoken) — OpenAI's own tokenizer library.
* [`mistral-common`](https://github.com/mistralai/mistral-common) — Mistral's official tokenizer package (relevant here since the Rennes server mostly hosts Mistral-family models).
* [`transformers`](https://huggingface.co/docs/transformers) — Hugging Face's library, which can load the tokenizer of pretty much any model published on the Hugging Face Hub; we'll use BERT's, which uses a different subword algorithm (**WordPiece**) than the other two (**BPE**).

All three run entirely locally once their (small) tokenizer files are downloaded — no OPENAI/MISTRAL/HF API key needed for this.

### OpenAI — `tiktoken`

In [ ]:
# Program 2: tokenizing a sentence with OpenAI's tiktoken

import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

text = "Tokenization is fascinating, but antidisestablishmentarianism is a mouthful."
token_ids = encoding.encode(text)

print(f"{len(text)} characters -> {len(token_ids)} tokens\n")
for token_id in token_ids:
    piece = encoding.decode([token_id])
    print(f"{token_id:>7}  {piece!r}")

We picked `cl100k_base` above, but `tiktoken` actually ships several named encodings, each tied to a specific generation of OpenAI models — you can list them all with `tiktoken.list_encoding_names()`. Here are the main ones and when each is used:

| Encoding | Used by | Notes |
|---|---|---|
| `gpt2` | GPT-2 | the original byte-level BPE, kept mostly for reference/compatibility |
| `r50k_base` | GPT-3 (`ada`, `babbage`, `curie`, `davinci`) | ~50k tokens vocabulary |
| `p50k_base` | Codex models, `text-davinci-002/003` | ~50k tokens, tuned for code |
| `p50k_edit` | `text-davinci-edit-001`, `code-davinci-edit-001` | for the (now retired) *edit* endpoint |
| `cl100k_base` | `gpt-3.5-turbo`, `gpt-4`, `text-embedding-ada-002`/`3-small`/`3-large` | ~100k tokens, the long-time default before GPT-4o |
| `o200k_base` | `gpt-4o`, `gpt-4.1`, `o1`/`o3`/`o4-mini`, the GPT-5(.x) family | ~200k tokens, the current default for most recent OpenAI models |
| `o200k_harmony` | `gpt-oss` (OpenAI's open-weight models) | pairs with the "harmony" response format |

In other words: which encoding applies depends entirely on *which model you're calling* — `tiktoken.encoding_for_model("gpt-4o")` picks `o200k_base` for you automatically, so you rarely need to name the encoding yourself unless, like we just did, you want to inspect a specific one directly.

*Source: [`tiktoken`'s own model-to-encoding mapping](https://github.com/openai/tiktoken/blob/main/tiktoken/model.py) (readable directly with `tiktoken.model.MODEL_TO_ENCODING`), cross-checked against [Tiktokenizer](https://tiktokenizer.com/) for the GPT-5.6 family.*

### Mistral — `mistral-common`

In [ ]:
# Program 3: tokenizing the same sentence with Mistral's mistral-common

from mistral_common.tokens.tokenizers.mistral import MistralTokenizer
from mistral_common.tokens.tokenizers.base import SpecialTokenPolicy

mistral_full_tokenizer = MistralTokenizer.v3()
mistral_tokenizer = mistral_full_tokenizer.instruct_tokenizer.tokenizer

mistral_ids = mistral_tokenizer.encode(text, bos=False, eos=False)

print(f"{len(text)} characters -> {len(mistral_ids)} tokens\n")
for token_id in mistral_ids:
    piece = mistral_tokenizer.decode([token_id], special_token_policy=SpecialTokenPolicy.IGNORE)
    print(f"{token_id:>7}  {piece!r}")

### Hugging Face — `transformers` (BERT, WordPiece)

Loading this tokenizer may print a couple of harmless messages: that PyTorch wasn't found (we don't need it, we're only loading a tokenizer, not a model) and a suggestion to set an `HF_TOKEN` for higher download rate limits (not needed for this public, ungated tokenizer, but see below if you want to set it up once and for all). Neither message is an error.

**Optional: getting an `HF_TOKEN`.** The warning is just about rate limits — everything works without it — but it's quick to set up and will matter more in Part 4, where we'll download an actual model (not just a tokenizer):
* Go to https://huggingface.co/join and create a free account (or log in if you already have one).
* Go to **Settings > Access Tokens** (https://huggingface.co/settings/tokens), click **Create new token**, choose the **Read** role (that's all we need), and copy it.
* Add a line in the `.env` file: `HF_TOKEN=<<copied token>>`

In [ ]:
# Program 4: tokenizing the same sentence with Hugging Face's transformers (BERT, WordPiece)

from dotenv import load_dotenv
from transformers import AutoTokenizer

load_dotenv(override=True)  # picks up HF_TOKEN from .env, if you set one (see above)

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_ids = bert_tokenizer.encode(text, add_special_tokens=False)

print(f"{len(text)} characters -> {len(bert_ids)} tokens\n")
for token_id in bert_ids:
    piece = bert_tokenizer.decode([token_id])
    print(f"{token_id:>7}  {piece!r}")

Unlike `tiktoken` or `mistral-common`, `transformers` isn't tied to one company's tokenizer: `AutoTokenizer.from_pretrained(...)` can load the tokenizer of essentially any model published on the Hugging Face Hub, each trained by its own author, sometimes with a different algorithm entirely. Here are a few well-known families:

| Model family (Hub example) | Algorithm | Notes |
|---|---|---|
| BERT (`bert-base-uncased`, used above) | WordPiece | lowercases everything, `##` continuation prefix |
| GPT-2 / RoBERTa (`gpt2`, `roberta-base`) | Byte-level BPE | same style as `tiktoken`'s own `gpt2` encoding |
| T5 / ALBERT / XLNet (`t5-small`, `albert-base-v2`) | Unigram (SentencePiece) | a different, probabilistic algorithm — picks the most *likely* segmentation instead of greedily merging |
| Llama / Mistral (`meta-llama/...`, `mistralai/...`) | SentencePiece BPE | same family as `mistral-common`, which we used directly above |
| Code models (`bigcode/starcoder2-3b`, `Salesforce/codegen-350M-mono`) | BPE trained on source code | vocabulary learned from programming languages rather than natural language — see below |

*Source: each model's own tokenizer configuration on the [Hugging Face Hub](https://huggingface.co/models) (visible in its `tokenizer_config.json`/`tokenizer.json`), and the [🤗 Tokenizers documentation](https://huggingface.co/docs/tokenizers) for the algorithm descriptions.*

The three tokenizers agree on the big picture — "Tokenization" gets split into a stem plus suffix, common short words like "is"/"a"/"but" mostly stay whole, and the deliberately obscure "antidisestablishmentarianism" gets broken into many pieces — but they disagree on the details:

* **OpenAI and Mistral** both use BPE, so they're similar in spirit, but they were trained on different data with a different vocabulary, so they don't produce the same token ids, nor exactly the same split (17 vs. 19 tokens on our example sentence). This directly matters in practice: the Rennes models we've been using since Part 1 are mostly Mistral-family models, so *their* token counts (and therefore their `max_tokens` limits and any per-token pricing) follow Mistral's tokenizer, not OpenAI's.
* **BERT's WordPiece** is a different algorithm: it lowercases everything ("Token" becomes "token"), and instead of encoding whether a piece starts a new word (like tiktoken's leading space), it marks *continuation* pieces with a `##` prefix (e.g. `##ization`) so you can tell "a new word starts here" from "this continues the previous piece".

Your turn to play: try encoding your own sentences above (a different language, a made-up word, some code, an emoji...) and see how each tokenizer splits it differently.

### Bonus: tokenizing code isn't like tokenizing text

So far we've only tokenized natural language. But tokenizers are trained on whatever corpus their author chose, and a tokenizer trained mostly on prose won't necessarily handle source code well: code has its own vocabulary (keywords, operators, indentation that carries meaning), which a text-only tokenizer never learned to merge efficiently — that's exactly why dedicated code tokenizers (like `p50k_base` above, or the `starcoder2`/`codegen` families) exist. Let's tokenize a small Python function with the two tokenizers we already have loaded, and see the difference.

In [ ]:
# Program 5: tokenizing a Python program instead of plain text

python_code = """def is_even(n):
    return n % 2 == 0
"""

print("--- tiktoken (cl100k_base) ---")
code_ids_tiktoken = encoding.encode(python_code)
print(f"{len(python_code)} characters -> {len(code_ids_tiktoken)} tokens")
for token_id in code_ids_tiktoken:
    print(f"{token_id:>7}  {encoding.decode([token_id])!r}")

print("\n--- BERT (WordPiece) ---")
code_ids_bert = bert_tokenizer.encode(python_code, add_special_tokens=False)
print(f"{len(python_code)} characters -> {len(code_ids_bert)} tokens")
for token_id in code_ids_bert:
    print(f"{token_id:>7}  {bert_tokenizer.decode([token_id])!r}")

The two results reveal very different priorities:

* **`tiktoken`** keeps the indentation as explicit tokens (a run of spaces, e.g. `'   '`), and treats operators like `==` and `%` as single tokens — both are semantically important in Python, where indentation defines code blocks. It also splits `is_even` into `is` + `_even`, since "is" is common in English but "is_even" as a whole isn't.
* **BERT's WordPiece** silently *drops the whitespace entirely* — no indentation, no newlines in the output — because its tokenizer was built for prose, where extra whitespace carries no meaning. It also splits `==` into two separate `=` tokens. For Python, where indentation is syntax, this would actually destroy information a model would need to understand the code correctly.

This is a good illustration of why "the vocabulary is learned, not universal" (our earlier takeaway) matters in practice: pick a tokenizer trained on the kind of content you're actually sending it — text, code, or something else — or you may lose information the model needs, quite apart from any effect on the token count.

## Control tokens: tokenizing a full prompt

<img src="images/control_tokens.jpg" width="150" alt="Control tokens" style="float: left; margin-right: 15px; margin-bottom: 10px;">

So far we've only tokenized a plain sentence. But since Part 1, what we actually send to an LLM isn't a plain string — it's a structured list of messages, each with a `role` (`system`, `user`, `assistant`) and some `content`, sometimes together with a list of `tools` the model is allowed to call. Before the model can read any of it, the tokenizer has to flatten this whole structure into a single sequence of tokens, and it does so using **control tokens** (also called special tokens): reserved ids that mark the start/end of the sequence and the boundary between roles (and tools), so the model can tell "this is an instruction" from "this is what the user said" from "here are the tools you can use".

Let's reuse a real example from Part 2: the `recruitment_agent`'s system instructions and its `send_message` tool (short enough to keep the token count readable — unlike `book_agent`, whose instructions contained the *entire* book, which would make this example explode into thousands of tokens), together with a plausible message from a student typing into the Gradio chat interface.

In [ ]:
# Program 6: tokenizing a full chat prompt (system + user + tool) with control tokens

from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.protocol.instruct.messages import SystemMessage, UserMessage
from mistral_common.protocol.instruct.tool_calls import Tool, Function

# Same recruitment_agent instructions as in Part 2 (Part_2_network_professor.ipynb).
recruitment_instructions = """
You want to recruit a student for the IoT course. Ask the student to provide their contact details, either their full name or their
email address. If you have either one, send a message to the course author using the send_message function.
"""

# Same send_message tool as in Part 2, described here the way the OpenAI Agents SDK
# would describe it to the model: a name, a description, and a JSON schema for its arguments.
send_message_tool = Tool(
    function=Function(
        name="send_message",
        description="Send a message to the course author to signal that a student is interested in the IoT course.",
        parameters={
            "type": "object",
            "properties": {
                "object": {"type": "string", "description": "The student's full name or email address."}
            },
            "required": ["object"],
        },
    )
)

# A chat request with a system message, a user message, and one available tool -
# exactly the kind of structure we've been passing to chat.completions.create() since Part 1.
request = ChatCompletionRequest(
    messages=[
        SystemMessage(content=recruitment_instructions),
        UserMessage(content="Hi, I'm Marie and I'd like to know more about the IoT course."),
    ],
    tools=[send_message_tool],
)

# encode_chat_completion() is what turns this whole structure (messages + tools)
# into the single flat sequence of token ids the model actually receives.
tokenized = mistral_full_tokenizer.encode_chat_completion(request)
tokens = tokenized.tokens
print(f"{len(tokens)} tokens total\n")

# Rather than dumping all 167 tokens, let's just show each control token
# together with a few tokens of context on each side, so it's easier to spot
# where the model was told "here are your tools" vs "here is your instruction".
context_size = 3

def pieces(ids):
    """Decode a list of token ids into their individual text pieces, for display."""
    return " ".join(
        repr(mistral_tokenizer.decode([t], special_token_policy=SpecialTokenPolicy.KEEP))
        for t in ids
    )

for i, token_id in enumerate(tokens):
    # is_special() tells us whether this token id is a reserved control token
    # (like <s>, [INST], [AVAILABLE_TOOLS]...) rather than an ordinary content token.
    if mistral_tokenizer.is_special(token_id):
        before = tokens[max(0, i - context_size):i]
        after = tokens[i + 1:i + 1 + context_size]
        control_piece = mistral_tokenizer.decode([token_id], special_token_policy=SpecialTokenPolicy.KEEP)
        print(f"... {pieces(before)}  >>> {control_piece!r} <<<  {pieces(after)} ...")

The tool made a real difference: right after `<s>`, a whole new pair of control tokens appears — `[AVAILABLE_TOOLS]` ... `[/AVAILABLE_TOOLS]` — wrapping the tool's JSON schema (`{"type": "function", "function": {"name": "send_message", ...}}`), *then* `[INST]` opens the actual instruction as before, and `[/INST]` closes it at the end. Notice also how much this costs in tokens: the tool's description and JSON syntax (braces, quotes, colons) alone account for the majority of the 167 tokens here, for a tool that does very little — this is exactly the "description of every tool the agent is allowed to call" we mentioned back in the introduction, and it adds up fast with several tools.

This is literally what happens on the server every time we call `chat.completions.create(messages=[...], tools=[...])`: our nice structured list of `{"role": ..., "content": ...}` dictionaries and tool definitions gets flattened into one sequence of tokens wrapped in control tokens like these. The model itself never sees a list of roles or a `tools` argument — it only ever sees a flat sequence of tokens.

And crucially: a token like `[AVAILABLE_TOOLS]` only means something because the model was specifically **trained** to recognize it. Chat/instruct models go through a fine-tuning stage on huge numbers of examples built exactly this way (`<s>[AVAILABLE_TOOLS]...[/AVAILABLE_TOOLS][INST]...[/INST]`), learning to treat everything between `[AVAILABLE_TOOLS]` and `[/AVAILABLE_TOOLS]` as tools it may call, and everything inside `[INST]`/`[/INST]` as an instruction to follow — not as more text to predict. A base model that never saw this training would just treat `[AVAILABLE_TOOLS]` as three more ordinary tokens, with no special meaning at all.

## Putting it all together: predicting one token at a time

<img src="images/predicting_loop.jpg" width="150" alt="Predicting loop" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Back in Part 1 we said an LLM works by repeatedly predicting the next token, appending it to everything that came before, and asking again — one token at a time — until the answer is complete. We can watch this happen directly by asking Ollama to *stream* its response: instead of waiting for the whole answer, the server sends us each newly predicted token the moment it's generated, so we can watch the sequence grow, one prediction at a time.

In [ ]:
# Program 7: watching token-by-token generation with Ollama streaming

from openai import OpenAI

# same Ollama setup as in Part 1 and Part 2, no API key needed
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

question = "In one short sentence, what is the Internet of Things?"

# stream=True keeps a single request open: instead of waiting for the full
# answer, Ollama sends each newly resolved token to us as soon as it's ready.
# The actual resolution (the model picking the next token, conditioned on
# everything before it) happens once, on the server, inside that one request.
stream = ollama.chat.completions.create(
    model="mistral:latest",
    messages=[{"role": "user", "content": question}],
    max_tokens=30,
    stream=True,
)

generated = ""
# Each `next_token` we receive here is a chunk the server has already
# resolved AND detokenized for us: what our loop observes, one iteration
# at a time, is really the incremental *detokenization* of an already
# ongoing generation - our code isn't driving the resolution step by step,
# it's just reading it off the wire as it comes in.
for next_token in stream:
    piece = next_token.choices[0].delta.content  # the decoded text for this chunk, or None
    if piece:
        generated += piece  # append this piece of text to what we've received so far
        print(f"+ {piece!r:15} -> {generated!r}")

Each line above is one step of the model's own resolution loop, running server-side: it looks at everything so far — the question, plus every token it has already produced — and resolves just the *next* one; that token gets appended, and the whole updated sequence is what conditions the following prediction, and so on, until we hit our `max_tokens` limit. Our Python `for` loop doesn't drive any of this — it just receives each already-resolved token, detokenized into text, as soon as the server produces it. This is exactly the mechanism described back in Part 1 — and now we've watched it happen, one token at a time.

<br>
<img src="images/autoregressive_academic.jpg" width="550" alt="Autoregressive Inherent Loop Diagram" style="display: block; margin: 15px auto;">
<br>

## Bonus: the raw mechanics, with a local PyTorch model

<img src="images/pytorch_torch.jpg" width="150" alt="PyTorch raw mechanics" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Program 7 let us *watch* the resolution loop happen, but Ollama did all the actual work on its own server — we only ever saw the already-decoded text. To see the mechanism itself, not just its result, we need direct access to the model: its raw output (called *logits*, one score per token in the vocabulary) and the choice of which token to pick next.

For this we'll load a small open model locally with `transformers` and `torch` — [`HuggingFaceTB/SmolLM2-135M`](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), only ~270MB and modest enough to run comfortably on a CPU, no GPU required. This is the same kind of setup we would reuse later to look at token *embeddings* directly.

In [ ]:
# Program 8: manually resolving one token at a time with a local model

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "HuggingFaceTB/SmolLM2-135M"
local_tokenizer = AutoTokenizer.from_pretrained(model_name)
local_model = AutoModelForCausalLM.from_pretrained(model_name)
local_model.eval()  # inference mode: disables things only needed for training

prompt = "In one short sentence, the Internet of Things is"

# Encode the prompt into token ids, as a batch of size 1 (that's what "return_tensors='pt'" gives us).
generated_ids = local_tokenizer.encode(prompt, return_tensors="pt")
generated_text = prompt

for i in range(12):
    # Run the model on everything generated so far (the whole point of this loop:
    # each step re-feeds the full sequence, exactly like we discussed in Part 1).
    with torch.no_grad():  # we're not training, so skip tracking gradients (faster, less memory)
        logits = local_model(generated_ids).logits

    # logits has one score per token in the vocabulary, for every position in the input;
    # [0, -1, :] takes the very last position, i.e. the scores for "what comes next".
    next_token_logits = logits[0, -1, :]

    # Turn those raw scores into probabilities that sum to 1.
    probabilities = torch.softmax(next_token_logits, dim=-1)

    # Greedy decoding: always pick the single most likely next token.
    next_token_id = torch.argmax(next_token_logits).item()
    confidence = probabilities[next_token_id].item()

    piece = local_tokenizer.decode([next_token_id])
    generated_text += piece

    # Append the newly resolved token id, so the next loop iteration sees it too.
    generated_ids = torch.cat([generated_ids, torch.tensor([[next_token_id]])], dim=1)

    print(f"+ {piece!r:12} (confidence={confidence:.2f}) -> {generated_text!r}")

Now we can see exactly what was hidden inside Ollama's server in Program 7: at every step, the model doesn't just output "the next token" — it outputs a *score for every single token in its vocabulary* (the `logits`), which `softmax` turns into a probability distribution, and greedy decoding just picks the one with the highest probability (its `confidence` above). Note the confidence swings a lot from step to step — very predictable continuations (like " of" after "a new way") score high, while more open-ended choices (like the very first word after our prompt) score much lower, since many different words could plausibly come next.

This is the *exact same loop* as Program 7 — encode, look at everything so far, resolve one token, append, repeat — except this time nothing is hidden: we're holding the logits ourselves. And since the model's vocabulary and its internal representations are tied together (each token id is really a row in a big matrix inside the model), this is also the setup we'd reuse to go one level deeper and look at token *embeddings* directly, in a later part.

## Tokens as the unit of billing

<img src="images/billing_soviet.jpg" width="150" alt="Tokens as unit of billing" style="float: left; margin-right: 15px; margin-bottom: 10px;">


Every paid LLM API bills you per token, not per character, word, or request — and crucially, **input and output tokens are priced separately**, with output usually costing noticeably more. Two real examples:

| Model | Input (per million tokens) | Output (per million tokens) |
|---|---|---|
| `gpt-5.6-luna` | $1.00 | $6.00 |
| `gemini-3.6-flash` (paid tier) | $1.50 | $7.50 |

Why is output so much pricier? Because of exactly what we saw in Programs 7 and 8: input tokens are all processed *at once*, in a single parallel pass over the whole prompt, while output tokens have to be resolved one at a time, autoregressively — each new token requires a full pass through the model, conditioned on everything before it. Generating is inherently more expensive, token for token, than reading.

This is also why we've been careful, since Part 2, about what goes into a prompt: every system instruction, every tool description, every bit of conversation history sent again because the model has no memory of it (Part 2's "amnesia") — all of that is billed as input tokens on every single call, whether or not the model actually needed it to answer.

*Source: [OpenAI's models & pricing page](https://developers.openai.com/api/docs/models) and [Google's Gemini API pricing page](https://ai.google.dev/gemini-api/docs/pricing).*

## The context window: a fixed-size memory

<img src="images/context_window_soviet.jpg" width="150" alt="The context window memory buffer" style="float: left; margin-right: 15px; margin-bottom: 10px;">


Every model also has a **context window**: a hard maximum on the number of tokens it can process in one go. `gpt-5.6-luna`, for example, is limited to about 1.05 million tokens. Crucially, this budget is shared by *everything* that goes through the model in a single call — the system instructions, the tool descriptions, the conversation history, the user's new message, **and** the tokens the model is about to generate as its answer. If the total goes over the limit, older content has to be dropped, or the call simply fails.

This is the real explanation behind Part 2's "amnesia": the model isn't choosing to forget — it has no memory at all outside of whatever tokens are inside its context window for *this* call. Resending the whole conversation history each time (as Part 2 suggested) works only until the history itself grows too large to fit in that window alongside everything else.

And, once again, this isn't a universal number: it depends on the model, exactly like its tokenizer's vocabulary. The local `mistral:latest` model we've been running with Ollama, for instance, natively supports a 32,768-token window — tiny compared to `gpt-5.6-luna`'s ~1 million — and Ollama may configure it with an even smaller effective window by default to save memory, unless you explicitly ask for more.

## Key takeaways

<img src="images/takeaways.jpg" width="150" alt="Key takeaways" style="float: left; margin-right: 15px; margin-bottom: 10px;">

* A token is neither a word nor a character: it's a subword unit, learned from data by an algorithm like BPE (or WordPiece), that keeps common words whole and splits rare ones into frequent pieces.
* This vocabulary is *learned*, not universal: different providers train their own tokenizer on their own data (sometimes even with a different algorithm), so the very same text can turn into a different number of tokens, and different token pieces, depending on which LLM you send it to.
* Beyond ordinary text tokens, the vocabulary also includes **control tokens** (like `<s>`, `[INST]`, `[/INST]`, `[AVAILABLE_TOOLS]`) that mark structure rather than words — this is how a list of `{"role": ..., "content": ...}` messages and tools becomes a single flat sequence the model can read, and it only works because chat/instruct models were specifically trained to understand it.
* Tokens are also the unit of everything else about an LLM API: **billing** (input and output tokens priced separately, output usually pricier) and the **context window** (a hard, per-model limit on how many tokens — input plus output combined — can fit in a single call).